In [1]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
with open('data/names.txt') as fobj:
    names = fobj.read().split('\n') # split file context at newline
# map letters to integer indices, including '.' (end-of-word character)
# this will help us build a transition matrix
alphabet = ".abcdefghijklmnopqrstuvwxyz"
idx = {letter:index for index, letter in enumerate(alphabet)}
idx

all_names = '.'+'.'.join(names)

Plan:
* Encode characters into one-hot vectors
* Stack together one-hot vectors into a window of a given length (here we'll use 3) to serve as the input data
* Train a NN with one hidden layer to predict next character

In [75]:
# set hyperparameters
window = 4
hidden_units = 200

# process data
# first convert letters to integer indices
all_idxs = torch.tensor([idx[c] for c in all_names])
# now convert to one-hot vectors
x_onehot = F.one_hot(all_idxs, num_classes=27).float()
x_onehot.shape # (num_samples, 27)

torch.Size([228146, 27])

In [76]:
x_onehot[:5], all_names[:5]

(tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0.]]),
 '.emma')

In [77]:
# now stack together windows of letters (i.e. rows of x_onehot)
N = x_onehot.shape[0]
# create a tensor to hold the data
X = torch.zeros((N-window, window, 27))
for k in range(N-window):
    for j in range(window):
        X[k, j] = x_onehot[k+j]

# get the Y vector that's aligned with this X
# should have first row of Y = 4th character of the sequence (i.e. index 3)
Y = all_idxs[window:]

In [78]:
X.shape, Y.shape

(torch.Size([228142, 4, 27]), torch.Size([228142]))

In [79]:
X[0] # 3 x 27 array encoding 3 characters, '.em' 

tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [80]:
Y[0] # single integer index encoding 'm' (the character following '.em')

tensor(1)

Construct the model:

* One hidden layer with `hidden_units` hidden units and a `tanh` nonlinearity
* Linear map to the output layer and a softmax activation
* Include bias at both layers

In [81]:
W1 = torch.randn((window * 27, hidden_units), requires_grad = True)
b1 = torch.randn((hidden_units,), requires_grad=True)

W2 = torch.randn((hidden_units, 27), requires_grad = True)
b2 = torch.randn((27,), requires_grad = True)

parameters = (W1, b1, W2, b2)

In [82]:
# need to reshape X so that it's (N, 81) rather than (N, 3, 27)
X.view(-1, 27*window).shape

torch.Size([228142, 108])

In [83]:
# run the model forward
hidden_activations = (X.view(-1, 27*window) @ W1 + b1).tanh() # tanh nonlinearity
# second layer
logits = hidden_activations @ W2 + b2

# compute the loss: same average negative log likelihood as we used last time
loss = F.cross_entropy(logits, Y)

In [84]:
loss.item()

22.364665985107422

In [85]:
# now let's train with gradient descent

h = .1 # step size
for _ in range(10):
    # clear the gradients
    for p in parameters:
        p.grad = None

    # run the model forward
    hidden_activations = (X.view(-1, 27*window) @ W1 + b1).tanh()
    logits = hidden_activations @ W2 + b2

    # compute loss
    loss = F.cross_entropy(logits, Y)
    print(loss.item())

    # gradient step
    loss.backward()
    for p in parameters:
        p.data -= h * p.grad    

22.364665985107422
21.750545501708984
21.20819091796875
20.72699546813965
20.29717254638672
19.910764694213867
19.561294555664062
19.24332046508789
18.952255249023438
18.684236526489258


In [107]:
# SGD
h = .01
for step in range(10000):
    # clear the gradients
    for p in parameters:
        p.grad = None

    # grab a minibatch of size 64
    batch_idxs = torch.randint(0, X.shape[0], (64,)) # low, high, shape
    hidden_activations = (X[batch_idxs, :, :].view(-1, 27*window) @ W1 + b1).tanh()
    logits = hidden_activations @ W2 + b2

    loss = F.cross_entropy(logits, Y[batch_idxs])
    if step%100 ==0:
        print(loss.item())

    loss.backward()
    for p in parameters:
        p.data -= h * p.grad    

# much faster!!

1.8839625120162964
2.2916419506073
2.173207998275757
2.2595579624176025
2.2775065898895264
2.265815258026123
2.0739073753356934
2.265007257461548
2.2289888858795166
2.233098030090332
2.1416966915130615
2.466297149658203
2.209404468536377
2.3104052543640137
2.2847232818603516
2.1666336059570312
2.052168369293213
2.3086650371551514
2.442258834838867
2.2105727195739746
2.6962010860443115
2.5061192512512207
2.032952070236206
2.1434876918792725
2.098412036895752
2.061185836791992
1.96333646774292
2.4157350063323975
2.2324650287628174
2.5116353034973145
2.2035741806030273
2.6433804035186768
2.3411142826080322
2.4103376865386963
2.0009164810180664
2.164381980895996
2.237208366394043
2.5024430751800537
2.3963844776153564
2.3748252391815186
2.27830171585083
2.1416478157043457
2.1589925289154053
2.4532415866851807
2.4687585830688477
2.2618868350982666
1.9399421215057373
2.0036866664886475
2.1138956546783447
2.2404799461364746
2.2860355377197266
2.3214969635009766
2.1706666946411133
2.47544741630

In [87]:
for p in parameters:
    print(f'{p.name}: mean={p.mean()}, var={p.std()}')

None: mean=0.007260570302605629, var=0.9550241231918335
None: mean=-0.09402629733085632, var=1.6867012977600098
None: mean=0.017503801733255386, var=0.6030415892601013
None: mean=0.27821412682533264, var=1.0854666233062744


In [88]:
# sample from the model!

def embed(word):
    idxs = torch.tensor([idx[c] for c in word])
    xenc = F.one_hot(idxs, num_classes=27).float()
    return xenc.view(-1, 27*len(word))

def generate(starting_word = "..."):
    word = starting_word
    done = False

    while not done:
        # grab the window of appropriate length
        context = embed(word[-window:])

        # apply the NN function
        hidden_activations = (context @ W1 + b1).tanh()
        logits = hidden_activations @ W2 + b2

        # convert outputs to probabilities (subtract max to ensure numerical stability) Do we actually have to do this?
        next_char_probs = F.softmax(logits - logits.max(), dim=1)
        # sample the next letter according to probs
        next_idx = torch.multinomial(next_char_probs, 1)
        next_char = alphabet[next_idx]
        if next_char == '.':
            done = True
        word = word + next_char
    return word


In [122]:
for _ in range(15):
    name = generate('..jo')
    print(name[:-1])

..joese
..joneet
..joive
..jovy
..jove
..joccriasar
..joekntes
..jorieni
..jolanna
..joiglet
..jodedy
..jorlla
..jovann
..jophw
..jopelida


In [70]:
# next: character embeddings